# Construction and evaluation of a SILVA 138.2 V3–V4 (341F/806R) taxonomic classifier

This notebook documents a reproducible workflow for constructing a **QIIME 2 / q2-feature-classifier Naive Bayes taxonomic classifier** specific to the V3–V4 region of the 16S rRNA gene, using the **SILVA 138.2 SSU Ref NR99** reference database and the primer pair **341F/806R**.

The workflow is organized into the following stages:

1. environment and resource preflight checks;
2. acquisition and validation of SILVA 138.2 SSU Ref NR99 data;
3. conversion of reference RNA sequences to DNA;
4. construction of fixed-rank SILVA taxonomy with RESCRIPt;
5. in silico extraction of the V3–V4 amplicon;
6. training of Naive Bayes classifiers;
7. classification of the training reference reads as a consistency check;
8. RESCRIPt evaluation of expected versus observed taxonomy;
9. provenance, caveats, and references.

> **Important methodological note.** The final evaluation in this notebook classifies the same reference sequences used to fit the classifier. This is a **resubstitution (training-set) evaluation** and is therefore optimistic. It is useful for detecting gross inconsistencies in the reference data, taxonomy, classifier serialization, and prediction workflow, but it must not be interpreted as an unbiased estimate of performance on independent biological samples.

### Scientific rationale

Taxonomic classification performance in marker-gene studies depends on the reference database, the amplified marker region, primer choice, taxonomic composition, and classifier configuration. Region-specific reference extraction can improve the correspondence between the reference representation and the amplicons observed experimentally. The QIIME 2 `q2-feature-classifier` plugin provides scikit-learn-based taxonomic classification, while RESCRIPt provides reproducible tools for acquiring, transforming, and evaluating reference sequence and taxonomy resources.

SILVA release **138.2** was released on **11 July 2024**. Its SSU Ref NR99 dataset is a non-redundant reference set constructed using a 99% identity criterion and is recommended by SILVA as a reference database for classification and phylogenetic applications.

**Primary references:** SILVA (Quast et al., 2013; Yilmaz et al., 2014), RESCRIPt (Robeson et al., 2021), and q2-feature-classifier (Bokulich et al., 2018).


## 1. Reproducibility configuration

All user-editable parameters are centralized in this section. Keeping paths, primer sequences, filenames, resource limits, and execution settings in one place reduces accidental inconsistencies between notebook cells.

The temporary directories are deliberately located under `/mnt/data`, because large QIIME 2 artifacts and Joblib memory-mapped arrays can require tens of gigabytes of temporary storage. The root filesystem on the target server is small relative to `/mnt/data`, so using `/tmp` may cause `OSError: [Errno 28] No space left on device`.

`JOBLIB_TEMP_FOLDER` is set separately because `classify-sklearn` uses Joblib/Loky for multiprocessing and may create large temporary memory-mapped files when `n_jobs > 1`.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import shlex
import datetime
import platform

# ---------------------------------------------------------------------
# Project configuration
# ---------------------------------------------------------------------

WORKDIR = Path("/mnt/data/qiime2-classifiers/silva-138.2-dev").resolve()

SILVA_VERSION = "138.2"
SILVA_TARGET = "SSURef_NR99"

F_PRIMER = "CCTACGGGRSGCAGCAG"
R_PRIMER = "GGACTACHVGGGTWTCTAAT"

MIN_AMPLICON_LENGTH = 350
MAX_AMPLICON_LENGTH = 550

N_JOBS_EXTRACT = 8

# Start conservatively. Increase only after confirming RAM and temporary-
# storage behavior on the server.
N_JOBS_CLASSIFY = 4
READS_PER_BATCH = 1000

TEMP_ROOT = Path("/mnt/data/tmp")
QIIME_TMP = TEMP_ROOT / "qiime2-lauro"
JOBLIB_TMP = TEMP_ROOT / "joblib-lauro"
QIIME_CACHE = TEMP_ROOT / "qiime2-cache-lauro"

# If True, commands are printed but not executed.
DRY_RUN = False

# If True, an existing expected output is accepted and the command is skipped.
# Set to False when a full rebuild is required.
SKIP_EXISTING = True

WORKDIR.mkdir(parents=True, exist_ok=True)

for directory in (QIIME_TMP, JOBLIB_TMP, QIIME_CACHE):
    directory.mkdir(parents=True, exist_ok=True)

os.environ.update({
    "TMPDIR": str(QIIME_TMP),
    "TMP": str(QIIME_TMP),
    "TEMP": str(QIIME_TMP),
    "JOBLIB_TEMP_FOLDER": str(JOBLIB_TMP),
})

os.chdir(WORKDIR)

print(f"Working directory      : {WORKDIR}")
print(f"QIIME temporary folder : {QIIME_TMP}")
print(f"Joblib temporary folder: {JOBLIB_TMP}")
print(f"QIIME cache            : {QIIME_CACHE}")


## 2. Utility functions

The original notebook executed every command directly with `!qiime`. That is convenient interactively, but it makes error checking, logging, output validation, and idempotent reruns harder.

The helpers below provide:

- explicit command construction without `shell=True`;
- timestamped log files;
- immediate failure on non-zero exit codes;
- optional skipping of already-created outputs;
- file existence checks;
- filesystem-space diagnostics;
- QIIME 2 artifact validation;
- reproducible recording of software versions.

Commands remain visible in the notebook output before execution.


In [ ]:
LOG_DIR = WORKDIR / "logs"
LOG_DIR.mkdir(exist_ok=True)

def human_bytes(n):
    units = ["B", "KiB", "MiB", "GiB", "TiB"]
    value = float(n)
    for unit in units:
        if value < 1024 or unit == units[-1]:
            return f"{value:.1f} {unit}"
        value /= 1024

def disk_report(path):
    path = Path(path)
    usage = shutil.disk_usage(path)
    return {
        "path": str(path),
        "total": human_bytes(usage.total),
        "used": human_bytes(usage.used),
        "free": human_bytes(usage.free),
        "free_bytes": usage.free,
    }

def require_file(path, min_size=1):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")
    if path.is_file() and path.stat().st_size < min_size:
        raise RuntimeError(f"File appears empty or truncated: {path}")
    return path

def outputs_exist(outputs):
    outputs = [Path(p) for p in outputs]
    return bool(outputs) and all(p.exists() and p.stat().st_size > 0 for p in outputs)

def run_command(args, *, outputs=None, log_name=None, skip_existing=None):
    """
    Execute a command robustly and capture stdout/stderr in a log file.

    Parameters
    ----------
    args : sequence[str | Path]
        Command and arguments.
    outputs : sequence[str | Path] | None
        Expected output files. If all exist and SKIP_EXISTING is enabled,
        the command is skipped.
    log_name : str | None
        Filename stem for the log.
    skip_existing : bool | None
        Override global SKIP_EXISTING.
    """
    args = [str(x) for x in args]
    outputs = [Path(p) for p in (outputs or [])]
    skip = SKIP_EXISTING if skip_existing is None else skip_existing

    if skip and outputs_exist(outputs):
        print("SKIP:", ", ".join(str(p.name) for p in outputs), "already exists.")
        return None

    printable = shlex.join(args)
    print(f"\n$ {printable}\n")

    if DRY_RUN:
        return None

    timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    stem = log_name or Path(args[0]).name
    log_path = LOG_DIR / f"{timestamp}_{stem}.log"

    env = os.environ.copy()
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.run(
            args,
            cwd=WORKDIR,
            env=env,
            stdout=log,
            stderr=subprocess.STDOUT,
            text=True,
            check=False,
        )

    # Show the tail of the log in the notebook, useful for successful
    # QIIME output and especially for failures.
    tail = log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-40:]
    if tail:
        print("\n".join(tail))

    if process.returncode != 0:
        raise RuntimeError(
            f"Command failed with exit code {process.returncode}. "
            f"See log: {log_path}"
        )

    for output in outputs:
        require_file(output)

    print(f"\nLog: {log_path}")
    return process

def qiime(*args, outputs=None, log_name=None, skip_existing=None):
    cmd = ["qiime", *map(str, args)]
    return run_command(
        cmd,
        outputs=outputs,
        log_name=log_name,
        skip_existing=skip_existing,
    )

def validate_artifact(path, level="max"):
    path = require_file(path)
    return qiime(
        "tools", "validate",
        str(path),
        "--level", level,
        log_name=f"validate_{path.stem}",
        skip_existing=False,
    )

def peek_artifact(path):
    path = require_file(path)
    return qiime(
        "tools", "peek",
        str(path),
        log_name=f"peek_{path.stem}",
        skip_existing=False,
    )


## 3. Environment and resource preflight

This section should be executed before expensive steps.

The classifier artifact is serialized with scikit-learn/Joblib. QIIME 2 warns that a `TaxonomicClassifier` trained with one scikit-learn version should not be assumed to be reliable under another version. Therefore, the exact software versions are recorded as part of the computational provenance.

For the QIIME 2 2026.7 distribution, the released environment specifies `q2-feature-classifier 2026.7.0` and `scikit-learn 1.7.1`. The local environment should nevertheless be queried directly rather than assumed.


In [ ]:
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print()

for p in [Path("/"), WORKDIR, QIIME_TMP, JOBLIB_TMP, QIIME_CACHE, Path("/dev/shm")]:
    if p.exists():
        r = disk_report(p)
        print(f"{r['path']:<40} free={r['free']:<12} total={r['total']}")

root_free = shutil.disk_usage("/").free
if root_free < 5 * 1024**3:
    print(
        "\nWARNING: the root filesystem has less than 5 GiB free. "
        "QIIME/Joblib temporary data are redirected to /mnt/data, but "
        "the root filesystem should still be cleaned to protect the OS "
        "and package-management operations."
    )

if shutil.disk_usage(QIIME_TMP).free < 50 * 1024**3:
    raise RuntimeError(
        f"Less than 50 GiB free under {QIIME_TMP}. "
        "Do not start classifier training/classification."
    )


In [ ]:
run_command(["qiime", "--version"], log_name="qiime_version", skip_existing=False)
run_command(
    ["python", "-c",
     "import sklearn, joblib; "
     "print('scikit-learn', sklearn.__version__); "
     "print('joblib', joblib.__version__)"],
    log_name="python_ml_versions",
    skip_existing=False,
)
run_command(
    ["python", "-c",
     "import tempfile, os; "
     "print('tempfile.gettempdir():', tempfile.gettempdir()); "
     "print('JOBLIB_TEMP_FOLDER:', os.getenv('JOBLIB_TEMP_FOLDER'))"],
    log_name="temporary_paths",
    skip_existing=False,
)


## 4. Reference data: SILVA 138.2 SSU Ref NR99

SILVA is a curated database of small- and large-subunit ribosomal RNA sequences. For release 138.2, SILVA describes **SSU Ref NR99** as a non-redundant SSU reference dataset generated with a 99% identity criterion. SILVA recommends this reduced reference collection for classification/phylogenetic reference use.

RESCRIPt provides `get-silva-data`, which can download, parse, and import SILVA directly. In this project, direct retrieval through that pipeline previously suffered interrupted/incomplete downloads. For that reason, the notebook retains a **manual-download fallback** with explicit integrity testing of the gzip file.

The manual sequence file used here is:

`SILVA_138.2_SSURef_NR99_tax_silva.fasta.gz`


In [ ]:
SILVA_FASTA_GZ = WORKDIR / "SILVA_138.2_SSURef_NR99_tax_silva.fasta.gz"

SILVA_FASTA_URL = (
    "https://www.arb-silva.de/fileadmin/silva_databases/"
    "release_138_2/Exports/SILVA_138.2_SSURef_NR99_tax_silva.fasta.gz"
)

# Optional preferred route when network stability permits:
#
# qiime(
#     "rescript", "get-silva-data",
#     "--p-version", SILVA_VERSION,
#     "--p-target", SILVA_TARGET,
#     "--o-silva-sequences", "silva-138.2-ssu-nr99-rna-seqs.qza",
#     "--o-silva-taxonomy", "silva-138.2-ssu-nr99-tax.qza",
#     outputs=[
#         "silva-138.2-ssu-nr99-rna-seqs.qza",
#         "silva-138.2-ssu-nr99-tax.qza",
#     ],
#     log_name="get_silva_data",
# )


In [ ]:
if not SILVA_FASTA_GZ.exists():
    run_command(
        [
            "wget",
            "--continue",
            "--tries=20",
            "--timeout=120",
            "--read-timeout=120",
            "--retry-connrefused",
            "--waitretry=10",
            "--output-document", SILVA_FASTA_GZ,
            SILVA_FASTA_URL,
        ],
        outputs=[SILVA_FASTA_GZ],
        log_name="download_silva_fasta",
    )
else:
    print(f"Using existing file: {SILVA_FASTA_GZ}")

# gzip -t performs an integrity test without decompressing the file.
run_command(
    ["gzip", "-t", SILVA_FASTA_GZ],
    log_name="test_silva_fasta_gzip",
    skip_existing=False,
)


### 4.1 Import SILVA RNA sequences into QIIME 2

The SILVA export is treated as `FeatureData[RNASequence]`. Importing it into a QIIME 2 artifact records the format transition in artifact provenance and makes the sequences available to RESCRIPt/QIIME 2 actions.


In [ ]:
RNA_QZA = WORKDIR / "silva-138.2-ssu-nr99-rna-seqs.qza"

qiime(
    "tools", "import",
    "--type", "FeatureData[RNASequence]",
    "--input-path", SILVA_FASTA_GZ,
    "--output-path", RNA_QZA,
    outputs=[RNA_QZA],
    log_name="import_silva_rna",
)

peek_artifact(RNA_QZA)
validate_artifact(RNA_QZA)


### 4.2 Reverse-transcribe the RNA reference into DNA

The `feature-classifier extract-reads` action operates on DNA sequence artifacts. RESCRIPt's `reverse-transcribe` action converts RNA sequence symbols to the corresponding DNA representation while preserving sequence identifiers.


In [ ]:
DNA_QZA = WORKDIR / "silva-138.2-ssu-nr99-dna-seqs.qza"

qiime(
    "rescript", "reverse-transcribe",
    "--i-rna-sequences", RNA_QZA,
    "--o-dna-sequences", DNA_QZA,
    "--use-cache", QIIME_CACHE,
    outputs=[DNA_QZA],
    log_name="reverse_transcribe_silva",
)

peek_artifact(DNA_QZA)
validate_artifact(DNA_QZA)


## 5. Construct fixed-rank SILVA taxonomy

`RESCRIPt parse-silva-taxonomy` combines three SILVA resources:

- the **taxonomic map**, linking sequence identifiers to SILVA taxonomic nodes;
- the **taxonomy/rank definition**;
- the **rooted SILVA taxonomy tree**.

RESCRIPt converts these data into a fixed-rank `FeatureData[Taxonomy]` representation suitable for downstream QIIME 2 classifiers.

Two taxonomies are built:

1. **six-rank taxonomy** through genus (`d__; p__; c__; o__; f__; g__`);
2. **seven-rank taxonomy** including species labels (`...; s__`).

> **Species-level caution.** QIIME 2 documentation explicitly warns that SILVA species annotations are included but are not curated at the species level to the same standard, and may therefore be unreliable. The species-level classifier in this notebook should be treated as exploratory and its assignments interpreted conservatively.


In [ ]:
TAXONOMY_BASE_URL = (
    "https://www.arb-silva.de/fileadmin/silva_databases/"
    "release_138_2/Exports/taxonomy"
)

taxonomy_downloads = {
    "taxmap_slv_ssu_ref_nr_138.2.txt.gz":
        f"{TAXONOMY_BASE_URL}/taxmap_slv_ssu_ref_nr_138.2.txt.gz",
    "tax_slv_ssu_138.2.txt.gz":
        f"{TAXONOMY_BASE_URL}/tax_slv_ssu_138.2.txt.gz",
    "tax_slv_ssu_138.2.tre.gz":
        f"{TAXONOMY_BASE_URL}/tax_slv_ssu_138.2.tre.gz",
}

for filename, url in taxonomy_downloads.items():
    target = WORKDIR / filename
    if target.exists():
        print(f"Using existing file: {target.name}")
        continue
    run_command(
        ["wget", "--continue", "--tries=20", "--output-document", target, url],
        outputs=[target],
        log_name=f"download_{Path(filename).stem}",
    )


In [ ]:
import gzip

def gunzip_keep_source(source):
    """Decompress .gz while preserving the compressed source file."""
    source = require_file(source)
    if source.suffix != ".gz":
        raise ValueError(f"Expected .gz file: {source}")

    destination = source.with_suffix("")
    if SKIP_EXISTING and destination.exists() and destination.stat().st_size > 0:
        print(f"SKIP: {destination.name} already exists.")
        return destination

    print(f"Decompressing {source.name} -> {destination.name}")
    with gzip.open(source, "rb") as fin, destination.open("wb") as fout:
        shutil.copyfileobj(fin, fout)

    require_file(destination)
    return destination

TAXMAP_TXT = gunzip_keep_source(WORKDIR / "taxmap_slv_ssu_ref_nr_138.2.txt.gz")
TAXRANK_TXT = gunzip_keep_source(WORKDIR / "tax_slv_ssu_138.2.txt.gz")
TAXTREE_NEWICK = gunzip_keep_source(WORKDIR / "tax_slv_ssu_138.2.tre.gz")


### 5.1 Import SILVA taxonomy components


In [ ]:
TAXMAP_QZA = WORKDIR / "taxmap-slv-ssu-ref-nr-138.2.qza"
TAXRANK_QZA = WORKDIR / "tax-slv-ssu-138.2.qza"
TAXTREE_QZA = WORKDIR / "tax-slv-ssu-138.2-tree.qza"

qiime(
    "tools", "import",
    "--type", "FeatureData[SILVATaxidMap]",
    "--input-path", TAXMAP_TXT,
    "--output-path", TAXMAP_QZA,
    outputs=[TAXMAP_QZA],
    log_name="import_silva_taxmap",
)

qiime(
    "tools", "import",
    "--type", "FeatureData[SILVATaxonomy]",
    "--input-path", TAXRANK_TXT,
    "--output-path", TAXRANK_QZA,
    outputs=[TAXRANK_QZA],
    log_name="import_silva_taxonomy_ranks",
)

qiime(
    "tools", "import",
    "--type", "Phylogeny[Rooted]",
    "--input-format", "NewickFormat",
    "--input-path", TAXTREE_NEWICK,
    "--output-path", TAXTREE_QZA,
    outputs=[TAXTREE_QZA],
    log_name="import_silva_taxonomy_tree",
)

for artifact in (TAXMAP_QZA, TAXRANK_QZA, TAXTREE_QZA):
    peek_artifact(artifact)


### 5.2 Parse genus-level and species-label taxonomies

The function below avoids duplicating nearly identical RESCRIPt commands and ensures that both outputs pass QIIME 2 validation.


In [ ]:
GENUS_TAX_QZA = WORKDIR / "silva-138.2-ssu-nr99-tax.qza"
SPECIES_TAX_QZA = WORKDIR / "silva-138.2-ssu-nr99-species-tax.qza"

def build_silva_taxonomy(output, include_species=False):
    args = [
        "rescript", "parse-silva-taxonomy",
        "--i-taxonomy-tree", TAXTREE_QZA,
        "--i-taxonomy-map", TAXMAP_QZA,
        "--i-taxonomy-ranks", TAXRANK_QZA,
    ]

    if include_species:
        args.append("--p-include-species-labels")

    args += [
        "--o-taxonomy", output,
        "--use-cache", QIIME_CACHE,
        "--verbose",
    ]

    qiime(
        *args,
        outputs=[output],
        log_name=f"parse_taxonomy_{'species' if include_species else 'genus'}",
    )
    peek_artifact(output)
    validate_artifact(output)

build_silva_taxonomy(GENUS_TAX_QZA, include_species=False)
build_silva_taxonomy(SPECIES_TAX_QZA, include_species=True)


## 6. In silico extraction of the V3–V4 amplicon

The classifier is trained on the same marker region targeted experimentally rather than on full-length SSU sequences.

Primer sequences:

- **341F:** `CCTACGGGRSGCAGCAG`
- **806R:** `GGACTACHVGGGTWTCTAAT`

The extraction allows amplicon lengths between **350 and 550 bp**, covering the expected V3–V4 product while tolerating biologically plausible length variation across reference sequences.

The `extract-reads` operation also produces an extraction-statistics artifact, which should be inspected to document how many reference sequences matched the primers and length constraints.


In [ ]:
V3V4_QZA = WORKDIR / "silva-138.2-v3v4-341f-806r-seqs.qza"
V3V4_STATS_QZA = WORKDIR / "silva-138.2-v3v4-341f-806r-stats.qza"

qiime(
    "feature-classifier", "extract-reads",
    "--i-sequences", DNA_QZA,
    "--p-f-primer", F_PRIMER,
    "--p-r-primer", R_PRIMER,
    "--p-read-orientation", "forward",
    "--p-min-length", str(MIN_AMPLICON_LENGTH),
    "--p-max-length", str(MAX_AMPLICON_LENGTH),
    "--p-n-jobs", str(N_JOBS_EXTRACT),
    "--o-read-extraction-stats", V3V4_STATS_QZA,
    "--o-reads", V3V4_QZA,
    "--use-cache", QIIME_CACHE,
    "--verbose",
    outputs=[V3V4_QZA, V3V4_STATS_QZA],
    log_name="extract_v3v4_341f_806r",
)

peek_artifact(V3V4_QZA)
validate_artifact(V3V4_QZA)

peek_artifact(V3V4_STATS_QZA)
validate_artifact(V3V4_STATS_QZA)


## 7. Train region-specific Naive Bayes classifiers

`fit-classifier-naive-bayes` fits a scikit-learn Naive Bayes pipeline to the extracted V3–V4 sequences and corresponding taxonomy.

The classifier artifact contains serialized scikit-learn objects. For reproducibility, use the classifier with the same compatible QIIME 2/scikit-learn software stack used for training.

Two classifiers are produced:

- **genus-oriented fixed-rank classifier**;
- **species-label classifier**, retained for exploratory comparison only because SILVA species labels require additional caution.


In [ ]:
GENUS_CLASSIFIER_QZA = WORKDIR / "silva-138.2-v3v4-341f-806r-nb-classifier.qza"
SPECIES_CLASSIFIER_QZA = WORKDIR / "silva-138.2-v3v4-341f-806r-nb-species-classifier.qza"

def train_classifier(reference_reads, reference_taxonomy, output):
    for required in (reference_reads, reference_taxonomy):
        require_file(required)

    qiime(
        "feature-classifier", "fit-classifier-naive-bayes",
        "--i-reference-reads", reference_reads,
        "--i-reference-taxonomy", reference_taxonomy,
        "--o-classifier", output,
        "--use-cache", QIIME_CACHE,
        "--verbose",
        outputs=[output],
        log_name=f"fit_{Path(output).stem}",
    )

    peek_artifact(output)
    validate_artifact(output)

train_classifier(V3V4_QZA, GENUS_TAX_QZA, GENUS_CLASSIFIER_QZA)
train_classifier(V3V4_QZA, SPECIES_TAX_QZA, SPECIES_CLASSIFIER_QZA)


## 8. Training-set classification as a consistency check

The reference amplicons are now classified using the classifier fitted from those same amplicons.

This is **not cross-validation** and it is **not an independent benchmark**. The purpose is narrower:

- confirm that the classifier can be loaded and executed;
- confirm compatibility between classifier and reference sequences;
- identify systematic disagreements between expected and predicted taxonomy;
- generate an internal consistency report before applying the classifier to experimental ASVs.

### Parallel execution and temporary storage

`classify-sklearn` uses Joblib/Loky when `n_jobs > 1`. Large feature matrices can be serialized as memory-mapped files. To prevent the server's `/tmp` or `/dev/shm` from filling, `JOBLIB_TEMP_FOLDER` is explicitly redirected to `/mnt/data/tmp/joblib-lauro`.

`reads_per_batch=1000` is intentionally conservative. QIIME 2 also supports automatic batch sizing; however, a fixed smaller batch is easier to reason about when memory/temporary-storage pressure has previously caused failures.

If a multiprocessing `PicklingError` wraps `OSError: [Errno 28] No space left on device`, first verify `JOBLIB_TEMP_FOLDER`, `TMPDIR`, and available space. For debugging, set `N_JOBS_CLASSIFY = 1`.


In [ ]:
GENUS_OBSERVED_QZA = WORKDIR / "silva-138.2-v3v4-341f-806r-nb-observed-taxonomy.qza"
SPECIES_OBSERVED_QZA = WORKDIR / "silva-138.2-v3v4-341f-806r-nb-species-observed-taxonomy.qza"

def classify_reference_reads(classifier, reads, output):
    require_file(classifier)
    require_file(reads)

    print("Temporary storage before classification:")
    for p in (QIIME_TMP, JOBLIB_TMP, QIIME_CACHE):
        r = disk_report(p)
        print(f"  {r['path']}: {r['free']} free")

    qiime(
        "feature-classifier", "classify-sklearn",
        "--i-classifier", classifier,
        "--i-reads", reads,
        "--p-n-jobs", str(N_JOBS_CLASSIFY),
        "--p-reads-per-batch", str(READS_PER_BATCH),
        "--o-classification", output,
        "--use-cache", QIIME_CACHE,
        "--verbose",
        outputs=[output],
        log_name=f"classify_{Path(output).stem}",
    )

    peek_artifact(output)
    validate_artifact(output)

classify_reference_reads(GENUS_CLASSIFIER_QZA, V3V4_QZA, GENUS_OBSERVED_QZA)
classify_reference_reads(SPECIES_CLASSIFIER_QZA, V3V4_QZA, SPECIES_OBSERVED_QZA)


## 9. Evaluate expected versus observed taxonomy

RESCRIPt `evaluate-classifications` compares the expected reference taxonomy with the taxonomy predicted by the fitted classifier and produces a `.qzv` visualization.

The resulting metrics should be described as **training-set/internal consistency metrics**, not as generalization performance.

For a publication-quality performance estimate, use an independent test set, cross-validation strategy, or taxonomically appropriate hold-out design. Particular care is required at species rank because marker-region resolution and reference-label quality can independently limit species discrimination.


In [ ]:
GENUS_EVAL_QZV = WORKDIR / "silva-138.2-v3v4-341f-806r-nb-evaluation.qzv"
SPECIES_EVAL_QZV = WORKDIR / "silva-138.2-v3v4-341f-806r-nb-species-evaluation.qzv"

def evaluate_classifier(expected_taxonomy, observed_taxonomy, output, label):
    require_file(expected_taxonomy)
    require_file(observed_taxonomy)

    qiime(
        "rescript", "evaluate-classifications",
        "--i-expected-taxonomies", expected_taxonomy,
        "--i-observed-taxonomies", observed_taxonomy,
        "--p-labels", label,
        "--o-evaluation", output,
        "--use-cache", QIIME_CACHE,
        "--verbose",
        outputs=[output],
        log_name=f"evaluate_{Path(output).stem}",
    )

    peek_artifact(output)
    validate_artifact(output)

evaluate_classifier(
    GENUS_TAX_QZA,
    GENUS_OBSERVED_QZA,
    GENUS_EVAL_QZV,
    "SILVA 138.2 V3-V4 341F-806R genus-rank training-set evaluation",
)

evaluate_classifier(
    SPECIES_TAX_QZA,
    SPECIES_OBSERVED_QZA,
    SPECIES_EVAL_QZV,
    "SILVA 138.2 V3-V4 341F-806R species-label training-set evaluation",
)


### Viewing `.qzv` results on a headless server

`qiime tools view` tries to launch a local graphical browser and is therefore generally inappropriate on a headless compute server.

Use one of the following approaches instead:

1. copy/download the `.qzv` artifact and open it with **QIIME 2 View** (`https://view.qiime2.org/`);
2. use a compatible Jupyter/QIIME 2 visualization integration if available in the environment;
3. retain the `.qzv` alongside the classifier as a provenance and quality-control artifact.

The cell below lists the final visualization files without attempting to launch a browser.


In [ ]:
for qzv in (GENUS_EVAL_QZV, SPECIES_EVAL_QZV):
    require_file(qzv)
    print(f"{qzv.name}: {human_bytes(qzv.stat().st_size)}")


## 10. Final artifact inventory and provenance checks

A classifier should not be distributed solely by filename. At minimum, retain:

- SILVA version and target collection;
- primer sequences and amplicon length limits;
- QIIME 2 / q2-feature-classifier / RESCRIPt versions;
- scikit-learn version;
- classifier `.qza`;
- extraction statistics;
- evaluation `.qzv`;
- this notebook and command logs.

QIIME 2 artifacts store provenance internally, but an external human-readable record remains useful for manuscripts, repositories, and collaborators.


In [ ]:
FINAL_ARTIFACTS = [
    V3V4_QZA,
    V3V4_STATS_QZA,
    GENUS_TAX_QZA,
    SPECIES_TAX_QZA,
    GENUS_CLASSIFIER_QZA,
    SPECIES_CLASSIFIER_QZA,
    GENUS_OBSERVED_QZA,
    SPECIES_OBSERVED_QZA,
    GENUS_EVAL_QZV,
    SPECIES_EVAL_QZV,
]

print(f"{'Artifact':75} {'Size':>12}")
print("-" * 90)
for artifact in FINAL_ARTIFACTS:
    if artifact.exists():
        print(f"{artifact.name:75} {human_bytes(artifact.stat().st_size):>12}")
    else:
        print(f"{artifact.name:75} {'MISSING':>12}")


## 11. Interpretation and limitations

### Region-specificity

This classifier is specialized for sequences compatible with the V3–V4 region defined by the 341F/806R primer pair and the extraction parameters in this notebook. A classifier trained in this way should not be assumed to be optimal for another 16S region or another primer pair.

### Database dependence

Taxonomic calls reflect the composition and annotation of SILVA 138.2 SSU Ref NR99. Reference-database updates can change sequence representation and taxonomic labels even when the experimental data do not change. Therefore, SILVA version **138.2** should be reported explicitly in manuscripts.

### Species-level interpretation

Species-level 16S assignments may be limited by:

- insufficient variation in the amplified V3–V4 region;
- identical or near-identical marker regions among related species;
- ambiguous or incomplete reference taxonomy;
- reliability of species annotations in the reference database.

The genus-level classifier should therefore be considered the primary taxonomic classifier unless the biological question and validation data justify species-level reporting.

### Training-set evaluation

The RESCRIPt evaluation generated above is a diagnostic of self-consistency. Because the training reads are also the evaluation reads, high scores are expected and cannot be used alone to claim high accuracy on real samples.

### Reproducibility

Serialized scikit-learn classifiers should be used with a compatible software environment. Record the QIIME 2 and scikit-learn versions and, ideally, preserve an environment specification or container definition with the classifier.


## 12. References

1. **Quast C, Pruesse E, Yilmaz P, et al.** The SILVA ribosomal RNA gene database project: improved data processing and web-based tools. *Nucleic Acids Research*. 2013;41(Database issue):D590–D596. https://doi.org/10.1093/nar/gks1219

2. **Yilmaz P, Parfrey LW, Yarza P, et al.** The SILVA and “All-species Living Tree Project (LTP)” taxonomic frameworks. *Nucleic Acids Research*. 2014;42(Database issue):D643–D648. https://doi.org/10.1093/nar/gkt1209

3. **Robeson MS II, O'Rourke DR, Kaehler BD, et al.** RESCRIPt: Reproducible sequence taxonomy reference database management. *PLoS Computational Biology*. 2021;17(11):e1009581. https://doi.org/10.1371/journal.pcbi.1009581

4. **Bokulich NA, Kaehler BD, Rideout JR, et al.** Optimizing taxonomic classification of marker-gene amplicon sequences with QIIME 2's q2-feature-classifier plugin. *Microbiome*. 2018;6:90. https://doi.org/10.1186/s40168-018-0470-z

5. **Pedregosa F, Varoquaux G, Gramfort A, et al.** Scikit-learn: Machine Learning in Python. *Journal of Machine Learning Research*. 2011;12:2825–2830.

6. **SILVA Release 138.2 documentation.** Release information dated 11 July 2024. https://www.arb-silva.de/documentation/release-1382/

7. **QIIME 2 q2-feature-classifier documentation.** Current plugin documentation and action reference. https://library.qiime2.org/plugins/qiime2/q2-feature-classifier/overview

8. **QIIME 2 / RESCRIPt documentation.** Plugin reference and SILVA parsing actions. https://library.qiime2.org/plugins/bokulich-lab/RESCRIPt

### Primer references used by the project

9. Primer 341F sequence associated with: https://doi.org/10.1371/journal.pone.0007401

10. Primer 806R sequence associated with: https://doi.org/10.1038/ismej.2012.8

> When preparing a manuscript, verify the exact primer attribution against the laboratory protocol and the primary sequencing-method paper, because primer names and sequence variants are reused across multiple 16S protocols.
